In [1]:
import pandas as pd

In [3]:
housing_price_df = pd.read_csv("Housing Price.csv")
housing_price_df.head()

,price,area,bedrooms,bathrooms,stories,mainroad,guestroom,basement,hotwaterheating,airconditioning,parking,prefarea,furnishingstatus
0,13300000,7420,4,2,3,yes,no,no,no,yes,2,yes,furnished
1,12250000,8960,4,4,4,yes,no,no,no,yes,3,no,furnished
2,12250000,9960,3,2,2,yes,no,yes,no,no,2,yes,semi-furnished
3,12215000,7500,4,2,2,yes,no,yes,no,yes,3,yes,furnished
4,11410000,7420,4,1,2,yes,yes,yes,no,yes,2,no,furnished


# Basic Cleaning and Feature Selection

In [11]:
# Drop rows with missing values (simple approach)
housing_price_df = housing_price_df.dropna()

# Separate features and target
X = housing_price_df.drop(columns=["price"])
y = housing_price_df["price"]

# Identify categorical columns
categorical_cols = X.select_dtypes(include=['object', 'bool']).columns

# Apply one-hot encoding to categorical columns
X = pd.get_dummies(X, columns=categorical_cols, drop_first=True)

In [8]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, PolynomialFeatures

In [12]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)


# Linear Regression

In [15]:
from sklearn.linear_model import LinearRegression

from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

X_single = X.iloc[:, 0].values.reshape(-1, 1)

X_train_s, X_test_s, y_train_s, y_test_s = train_test_split(
    X_single, y, test_size=0.2, random_state=42
)

model_lr = LinearRegression()
model_lr.fit(X_train_s, y_train_s)

y_pred = model_lr.predict(X_test_s)

print("Linear Regression R2:", r2_score(y_test_s, y_pred))


Linear Regression R2: 0.27287851871974644


# Multiple Linear Regression

In [16]:
model_mlr = LinearRegression()
model_mlr.fit(X_train_scaled, y_train)

y_pred = model_mlr.predict(X_test_scaled)

print("Multiple Linear Regression")
print("MAE:", mean_absolute_error(y_test, y_pred))
print("MSE:", mean_squared_error(y_test, y_pred))
print("R2:", r2_score(y_test, y_pred))


Multiple Linear Regression
MAE: 970043.4039201642
MSE: 1754318687330.6677
R2: 0.6529242642153177


# Polynomial Regression

In [17]:
poly = PolynomialFeatures(degree=2)
X_train_poly = poly.fit_transform(X_train_scaled)
X_test_poly = poly.transform(X_test_scaled)

model_poly = LinearRegression()
model_poly.fit(X_train_poly, y_train)

y_pred = model_poly.predict(X_test_poly)

print("Polynomial Regression (Degree 2)")
print("MSE:", mean_squared_error(y_test, y_pred))
print("R2:", r2_score(y_test, y_pred))


Polynomial Regression (Degree 2)
MSE: 1901686413946.4487
R2: 0.6237689217365155


# KNN Regression

In [19]:
from sklearn.neighbors import KNeighborsRegressor

model_knn = KNeighborsRegressor(
    n_neighbors=5,
    weights="distance"
)

model_knn.fit(X_train_scaled, y_train)

y_pred = model_knn.predict(X_test_scaled)

print("KNN Regression")
print("MSE:", mean_squared_error(y_test, y_pred))
print("R2:", r2_score(y_test, y_pred))


KNN Regression
MSE: 1943324338523.095
R2: 0.6155312432500738


# Decision Tree Regression

In [20]:
from sklearn.tree import DecisionTreeRegressor

model_dt = DecisionTreeRegressor(
    max_depth=5,
    random_state=42
)

model_dt.fit(X_train, y_train)

y_pred = model_dt.predict(X_test)

print("Decision Tree Regression")
print("MSE:", mean_squared_error(y_test, y_pred))
print("R2:", r2_score(y_test, y_pred))


Decision Tree Regression
MSE: 2701167171509.852
R2: 0.46559904406211106


# Model Comparison

In [21]:
import matplotlib.pyplot as plt
import seaborn as sns

models = {
    "Multiple Linear": model_mlr,
    "Polynomial": model_poly,
    "KNN": model_knn,
    "Decision Tree": model_dt
}

for name, model in models.items():
    if name == "Polynomial":
        preds = model.predict(X_test_poly)
    elif name == "Decision Tree":
        preds = model.predict(X_test)
    else:
        preds = model.predict(X_test_scaled)

    print(f"{name} R2:", r2_score(y_test, preds))


Multiple Linear R2: 0.6529242642153177
Polynomial R2: 0.6237689217365155
KNN R2: 0.6155312432500738
Decision Tree R2: 0.46559904406211106
